## Kütüphaneleri Import Etme

In [1]:
import pandas as pd
import numpy as np
from sentence_transformers import SentenceTransformer
import pyarrow as pa

C:\Users\kaptan\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\LocalCache\local-packages\Python312\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Embedding Modeli Yükleme

In [2]:
model = SentenceTransformer('all-MiniLM-L6-v2')

## Config Ayarları

In [3]:
metadata_path = '../data/processed/vector_metadata.parquet'

emb_path = '../data/processed/metadata_embeddings.npy'
ids_path = '../data/processed/ids_embeddings.npy'

## Veri Dosyası Okuma

In [4]:
df = pd.read_parquet(metadata_path, engine='pyarrow')

In [5]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 2 columns):
 #   Column         Non-Null Count  Dtype 
---  ------         --------------  ----- 
 0   recipe_id      10000 non-null  int32 
 1   embedded_text  10000 non-null  object
dtypes: int32(1), object(1)
memory usage: 117.3+ KB


## Verilerin Vektörizasyonu
Elimizdeki verileri vektörize edelim ve hızlı bir başlangıç için test verisi oluşturalım.

In [6]:
# Encode full dataset into a single (N, D) float32 numpy array
embs = model.encode(
    df['embedded_text'].fillna('').tolist(),
    show_progress_bar=True,
    convert_to_numpy=True,
    batch_size=128).astype('float32')

# L2-normalize embeddings (recommended) and avoid zero-division
norms = np.linalg.norm(embs, axis=1, keepdims=True)
norms[norms == 0] = 1.0
embs = embs / norms

# Save normalized embeddings and ids as .npy files (use mmap on load)
ids = np.asarray(df['recipe_id'].to_numpy(), dtype=np.int64)

np.save(emb_path, embs)
np.save(ids_path, ids)

Batches: 100%|██████████| 79/79 [13:26<00:00, 10.21s/it]
